# 🏢 Employee Activity Detection
### End-to-End Pipeline: YOLOv8n (Person Detection) + EfficientNet-B0 (Activity Classification)

This notebook covers the full workflow:
1. Environment setup & GPU check
2. Dataset upload & preprocessing
3. EfficientNet-B0 training on the HAR dataset
4. Model evaluation & metrics
5. Building the two-stage inference pipeline
6. Running inference on video / webcam feed
7. Saving & exporting models

**Dataset expected:** `archive.zip` containing:
```
Human Action Recognition/
  Training_set.csv
  Testing_set.csv
  train/  (Image_*.jpg)
  test/   (Image_*.jpg)
```

## ⚙️ Cell 1 — Environment Setup & GPU Check

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q ultralytics timm albumentations opencv-python-headless

import torch, os, time, random, warnings
warnings.filterwarnings('ignore')

# GPU check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Device: {device}")
if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be slow. Enable GPU: Runtime > Change runtime type > T4 GPU")

## 📦 Cell 2 — Upload & Extract Dataset

In [ ]:
# ── Option A: Upload from local machine ──────────────────────────────────────
from google.colab import files
print("Upload your archive.zip (the HAR dataset)...")
uploaded = files.upload()   # Select your archive.zip

import zipfile, pathlib

ZIP_NAME = list(uploaded.keys())[0]
DATA_ROOT = pathlib.Path('/content/dataset')

with zipfile.ZipFile(ZIP_NAME, 'r') as z:
    z.extractall(DATA_ROOT)

print(f"✅ Extracted to {DATA_ROOT}")

# ── Option B: Load from Google Drive (comment out Option A, uncomment below) ─
# from google.colab import drive
# drive.mount('/content/drive')
# ZIP_PATH = '/content/drive/MyDrive/archive.zip'   # ← adjust path
# DATA_ROOT = pathlib.Path('/content/dataset')
# with zipfile.ZipFile(ZIP_PATH, 'r') as z:
#     z.extractall(DATA_ROOT)
# print(f'✅ Extracted to {DATA_ROOT}')

## 🗂️ Cell 3 — Dataset Exploration & Paths

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ── Locate CSV and image directories ─────────────────────────────────────────
HAR_DIR   = DATA_ROOT / 'Human Action Recognition'
TRAIN_CSV = HAR_DIR / 'Training_set.csv'
TEST_CSV  = HAR_DIR / 'Testing_set.csv'
TRAIN_IMG = HAR_DIR / 'train'
TEST_IMG  = HAR_DIR / 'test'

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f"Training samples : {len(train_df)}")
print(f"Test samples     : {len(test_df)}")
print(f"\nClasses ({train_df['label'].nunique()} total):")
print(sorted(train_df['label'].unique()))

# ── Class distribution ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
counts = train_df['label'].value_counts().sort_values(ascending=True)
counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Training Set — Samples per Activity Class')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

# ── Sample images grid ────────────────────────────────────────────────────────
CLASSES = sorted(train_df['label'].unique())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
NUM_CLASSES  = len(CLASSES)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for ax, cls in zip(axes.flatten(), CLASSES):
    sample = train_df[train_df['label'] == cls].iloc[0]
    img_path = TRAIN_IMG / sample['filename']
    img = mpimg.imread(str(img_path))
    ax.imshow(img)
    ax.set_title(cls.replace('_', '\n'), fontsize=8)
    ax.axis('off')
plt.suptitle('One sample per activity class', y=1.01)
plt.tight_layout()
plt.show()

print(f"\nClass → Index mapping:")
for c, i in CLASS_TO_IDX.items():
    print(f"  {i:2d}  {c}")

## 🔄 Cell 4 — Dataset & DataLoaders

In [ ]:
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image

IMG_SIZE   = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# ── Augmentation pipelines ────────────────────────────────────────────────────
train_transforms = A.Compose([
    A.SmallestMaxSize(max_size=256),
    A.RandomCrop(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.5),
    A.GaussNoise(var_limit=(10, 50), p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.SmallestMaxSize(max_size=256),
    A.CenterCrop(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class ActivityDataset(Dataset):
    def __init__(self, df, img_dir, class_to_idx, transforms=None, has_labels=True):
        self.df           = df.reset_index(drop=True)
        self.img_dir      = pathlib.Path(img_dir)
        self.class_to_idx = class_to_idx
        self.transforms   = transforms
        self.has_labels   = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = self.img_dir / row['filename']
        image    = np.array(Image.open(img_path).convert('RGB'))

        if self.transforms:
            image = self.transforms(image=image)['image']

        if self.has_labels:
            label = self.class_to_idx[row['label']]
            return image, label
        return image, row['filename']

# ── Split training → train / validation (90/10) ───────────────────────────────
train_split, val_split = train_test_split(
    train_df, test_size=0.10, stratify=train_df['label'], random_state=42
)

train_dataset = ActivityDataset(train_split, TRAIN_IMG, CLASS_TO_IDX, train_transforms)
val_dataset   = ActivityDataset(val_split,   TRAIN_IMG, CLASS_TO_IDX, val_transforms)
test_dataset  = ActivityDataset(test_df,     TEST_IMG,  CLASS_TO_IDX, val_transforms, has_labels=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train  : {len(train_dataset):,} samples  ({len(train_loader)} batches)")
print(f"Val    : {len(val_dataset):,} samples  ({len(val_loader)} batches)")
print(f"Test   : {len(test_dataset):,} samples  ({len(test_loader)} batches)")

## 🧠 Cell 5 — EfficientNet-B0 Model Definition

In [ ]:
import timm
import torch.nn as nn

def build_model(num_classes, pretrained=True):
    """EfficientNet-B0 with custom classification head."""
    model = timm.create_model(
        'efficientnet_b0',
        pretrained=pretrained,
        num_classes=0,          # Remove default head
        global_pool='avg'
    )
    in_features = model.num_features  # 1280 for B0

    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 512),
        nn.SiLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes)
    )
    return model


def freeze_backbone(model):
    """Freeze all layers except the classifier head."""
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False


def unfreeze_backbone(model):
    """Unfreeze all layers for full fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True


model = build_model(NUM_CLASSES).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model       : EfficientNet-B0 (timm)")
print(f"Total params: {total_params:,}")
print(f"Trainable   : {trainable_params:,}")
print(f"Classes     : {NUM_CLASSES}")

## 🏋️ Cell 6 — Training Loop

In [ ]:
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import OneCycleLR

# ── Training hyperparameters ──────────────────────────────────────────────────
WARMUP_EPOCHS  = 5      # Frozen backbone, train head only
FINETUNE_EPOCHS = 20    # Full fine-tune (all layers)
LR_WARMUP      = 1e-3
LR_FINETUNE    = 5e-5
CHECKPOINT_DIR = pathlib.Path('/content/checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ── Class weights to handle any imbalance ─────────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array(CLASSES),
    y=train_split['label'].values
)
weights_tensor = torch.tensor(
    [class_weights[CLASS_TO_IDX[c]] for c in CLASSES],
    dtype=torch.float32
).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

# ── Helper: one epoch ─────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer, scheduler, scaler, is_train):
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            with autocast():
                outputs = model(images)
                loss    = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                if scheduler: scheduler.step()

            total_loss += loss.item() * images.size(0)
            preds       = outputs.argmax(1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    return total_loss / total, correct / total


# ── Training orchestrator ─────────────────────────────────────────────────────
def train(model, warmup_epochs, finetune_epochs):
    scaler    = GradScaler()
    history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0

    # ── Phase 1: Warmup (frozen backbone) ─────────────────────────────────────
    print("\n📌 Phase 1: Warmup — training head only")
    freeze_backbone(model)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR_WARMUP
    )
    scheduler = OneCycleLR(
        optimizer, max_lr=LR_WARMUP,
        steps_per_epoch=len(train_loader), epochs=warmup_epochs
    )

    for epoch in range(1, warmup_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, scheduler, scaler, True)
        vl_loss, vl_acc = run_epoch(model, val_loader,   optimizer, None,      scaler, False)
        history['train_loss'].append(tr_loss);  history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss);    history['val_acc'].append(vl_acc)
        print(f"  Warmup {epoch:02d}/{warmup_epochs} | "
              f"Loss {tr_loss:.4f}/{vl_loss:.4f} | "
              f"Acc {tr_acc:.3f}/{vl_acc:.3f} | "
              f"{time.time()-t0:.1f}s")

    # ── Phase 2: Full fine-tune ───────────────────────────────────────────────
    print("\n🔥 Phase 2: Full fine-tune — all layers")
    unfreeze_backbone(model)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR_FINETUNE, weight_decay=1e-4
    )
    scheduler = OneCycleLR(
        optimizer, max_lr=LR_FINETUNE,
        steps_per_epoch=len(train_loader), epochs=finetune_epochs
    )

    for epoch in range(1, finetune_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, scheduler, scaler, True)
        vl_loss, vl_acc = run_epoch(model, val_loader,   optimizer, None,      scaler, False)
        history['train_loss'].append(tr_loss);  history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss);    history['val_acc'].append(vl_acc)

        tag = ""
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), CHECKPOINT_DIR / 'best_efficientnet_b0.pth')
            tag = "  ✅ saved"

        print(f"  Finetune {epoch:02d}/{finetune_epochs} | "
              f"Loss {tr_loss:.4f}/{vl_loss:.4f} | "
              f"Acc {tr_acc:.3f}/{vl_acc:.3f} | "
              f"{time.time()-t0:.1f}s{tag}")

    print(f"\n✅ Best val accuracy: {best_val_acc:.4f}")
    return history


history = train(model, WARMUP_EPOCHS, FINETUNE_EPOCHS)

## 📈 Cell 7 — Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs, history['train_loss'], label='Train', color='steelblue')
ax1.plot(epochs, history['val_loss'],   label='Val',   color='tomato')
ax1.axvline(WARMUP_EPOCHS, color='gray', linestyle='--', alpha=0.6, label='Warmup end')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(epochs, history['train_acc'], label='Train', color='steelblue')
ax2.plot(epochs, history['val_acc'],   label='Val',   color='tomato')
ax2.axvline(WARMUP_EPOCHS, color='gray', linestyle='--', alpha=0.6, label='Warmup end')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()

plt.suptitle('EfficientNet-B0 Training Curves')
plt.tight_layout()
plt.show()

## 📊 Cell 8 — Evaluation: Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ── Load best checkpoint ──────────────────────────────────────────────────────
model.load_state_dict(torch.load(CHECKPOINT_DIR / 'best_efficientnet_b0.pth',
                                  map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds   = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# ── Classification report ─────────────────────────────────────────────────────
print(classification_report(
    all_labels, all_preds,
    target_names=[IDX_TO_CLASS[i] for i in range(NUM_CLASSES)],
    digits=3
))

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=[IDX_TO_CLASS[i] for i in range(NUM_CLASSES)],
    yticklabels=[IDX_TO_CLASS[i] for i in range(NUM_CLASSES)],
    ax=ax
)
ax.set_title('Normalised Confusion Matrix — Validation Set')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 🔍 Cell 9 — YOLOv8n Person Detector Setup

In [ ]:
from ultralytics import YOLO
import cv2

# Download YOLOv8n (nano) — pre-trained on COCO, class 0 = 'person'
yolo_model = YOLO('yolov8n.pt')

# Quick sanity check on a sample image
sample_path = str(list((TRAIN_IMG).glob('*.jpg'))[0])
results = yolo_model(sample_path, classes=[0], conf=0.4, verbose=False)

img_bgr = cv2.imread(sample_path)
for r in results:
    for box in r.boxes.xyxy.cpu().numpy().astype(int):
        cv2.rectangle(img_bgr, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(6, 6))
plt.imshow(img_rgb)
plt.title('YOLOv8n — person detection test')
plt.axis('off')
plt.show()
print(f"✅ Persons detected: {len(results[0].boxes)}")

## 🔧 Cell 10 — Two-Stage Inference Pipeline

In [ ]:
import torchvision.transforms.functional as TF

# ── Colour palette for bounding box labels ────────────────────────────────────
np.random.seed(42)
CLASS_COLORS = {
    cls: tuple(np.random.randint(50, 255, 3).tolist())
    for cls in CLASSES
}

# ── Preprocessing for EfficientNet-B0 (inference) ────────────────────────────
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def preprocess_crop(crop_bgr):
    """BGR crop → normalised tensor (1, 3, 224, 224)."""
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    crop_rgb = cv2.resize(crop_rgb, (IMG_SIZE, IMG_SIZE))
    tensor   = (crop_rgb / 255.0 - MEAN) / STD
    tensor   = torch.tensor(tensor, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)
    return tensor.to(device)


def classify_crop(model, crop_bgr, topk=1):
    """Return (label, confidence) for a person crop."""
    tensor = preprocess_crop(crop_bgr)
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0]
    top_probs, top_idxs = probs.topk(topk)
    return [
        (IDX_TO_CLASS[idx.item()], prob.item())
        for idx, prob in zip(top_idxs, top_probs)
    ]


def annotate_frame(frame_bgr, yolo_model, effnet_model,
                   yolo_conf=0.4, cls_conf_threshold=0.4,
                   padding=20):
    """
    Full pipeline on a single BGR frame.
    Returns annotated BGR frame + list of detected activities.
    """
    H, W = frame_bgr.shape[:2]
    results  = yolo_model(frame_bgr, classes=[0], conf=yolo_conf, verbose=False)
    detected = []

    for r in results:
        for box in r.boxes.xyxy.cpu().numpy().astype(int):
            x1, y1, x2, y2 = box

            # Pad crop slightly for context
            cx1 = max(0, x1 - padding)
            cy1 = max(0, y1 - padding)
            cx2 = min(W, x2 + padding)
            cy2 = min(H, y2 + padding)
            crop = frame_bgr[cy1:cy2, cx1:cx2]

            if crop.size == 0:
                continue

            # Classify
            top = classify_crop(effnet_model, crop, topk=1)[0]
            label, conf = top
            detected.append({'bbox': (x1,y1,x2,y2), 'label': label, 'conf': conf})

            # Draw
            color = CLASS_COLORS[label]
            cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color, 2)

            text   = f"{label.replace('_',' ')} {conf:.0%}"
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(frame_bgr, (x1, y1-th-8), (x1+tw+6, y1), color, -1)
            cv2.putText(frame_bgr, text, (x1+3, y1-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 2)

    # FPS counter placeholder (filled in the video loop)
    cv2.putText(frame_bgr, f"Persons: {len(detected)}", (10, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

    return frame_bgr, detected


print("✅ Inference pipeline ready.")

## 🖼️ Cell 11 — Test Pipeline on Sample Images

In [ ]:
# Re-load best weights (in case cell order matters)
model.load_state_dict(torch.load(CHECKPOINT_DIR / 'best_efficientnet_b0.pth',
                                  map_location=device))
model.eval()

# ── Run on 6 random validation images ─────────────────────────────────────────
sample_files = random.sample(list(val_split['filename']), 6)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, fname in zip(axes.flatten(), sample_files):
    frame = cv2.imread(str(TRAIN_IMG / fname))
    annotated, detections = annotate_frame(frame.copy(), yolo_model, model)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))

    true_label = val_split.loc[val_split['filename']==fname, 'label'].values[0]
    pred_label = detections[0]['label'] if detections else 'no detection'
    color      = 'green' if true_label == pred_label else 'red'
    ax.set_title(f"True: {true_label}\nPred: {pred_label}",
                 color=color, fontsize=9)
    ax.axis('off')

plt.suptitle('Pipeline Test — Green = Correct, Red = Wrong', fontsize=13)
plt.tight_layout()
plt.show()

## 🎬 Cell 12 — Process a Video File

In [ ]:
# ── Upload a video ────────────────────────────────────────────────────────────
print("Upload an MP4/AVI video file (office footage):")
uploaded_video = files.upload()
VIDEO_PATH  = list(uploaded_video.keys())[0]
OUTPUT_PATH = '/content/output_annotated.mp4'

cap  = cv2.VideoCapture(VIDEO_PATH)
fps  = int(cap.get(cv2.CAP_PROP_FPS)) or 25
W    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (W, H))

print(f"Processing {total_frames} frames at {fps} FPS ({W}×{H})...")

frame_times = []
frame_idx   = 0
model.eval()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    t0 = time.time()
    annotated, detections = annotate_frame(frame, yolo_model, model)
    elapsed = time.time() - t0
    frame_times.append(elapsed)

    # FPS overlay
    cv2.putText(annotated, f"FPS: {1/elapsed:.1f}", (10, 58),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

    out.write(annotated)
    frame_idx += 1
    if frame_idx % 50 == 0:
        avg_fps = 1 / np.mean(frame_times[-50:])
        print(f"  Frame {frame_idx}/{total_frames}  |  avg {avg_fps:.1f} FPS")

cap.release()
out.release()

avg_fps = 1 / np.mean(frame_times)
print(f"\n✅ Done! Average speed: {avg_fps:.1f} FPS")
print(f"Output saved to: {OUTPUT_PATH}")
files.download(OUTPUT_PATH)

## 📹 Cell 13 — Live Webcam Demo (Colab)

In [ ]:
from IPython.display import display, Image as IPyImage, clear_output
from google.colab.output import eval_js
from base64 import b64decode
import time

def b64_to_frame(data_url):
    img_bytes = b64decode(data_url.split(',')[1])
    nparr = np.frombuffer(img_bytes, dtype=np.uint8)
    return cv2.imdecode(nparr, cv2.IMREAD_COLOR)

# Single self-contained JS: opens camera, waits for it to be ready,
# grabs the frame, closes the stream, returns base64 — all in one eval_js call.
CAPTURE_JS = """
new Promise(async (resolve, reject) => {
    try {
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        const video  = document.createElement('video');
        video.srcObject = stream;
        video.style.display = 'none';
        document.body.appendChild(video);

        // Wait until the video has actual dimensions
        await new Promise(r => {
            video.onloadedmetadata = () => { video.play().then(r); };
        });
        // Extra settling time for exposure/focus
        await new Promise(r => setTimeout(r, 800));

        const canvas = document.createElement('canvas');
        canvas.width  = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);

        stream.getTracks().forEach(t => t.stop());
        video.remove();

        resolve(canvas.toDataURL('image/jpeg', 0.85));
    } catch(e) {
        reject(e.toString());
    }
});
"""

NUM_FRAMES  = 5
FRAME_DELAY = 2.0   # seconds between captures (camera re-opens each time)

model.eval()
print(f"Capturing {NUM_FRAMES} frames — allow camera access when prompted...\n")

for i in range(NUM_FRAMES):
    print(f"Frame {i+1}/{NUM_FRAMES} — opening camera...")
    data_url  = eval_js(CAPTURE_JS)
    frame_bgr = b64_to_frame(data_url)

    t0 = time.time()
    annotated, detections = annotate_frame(frame_bgr.copy(), yolo_model, model)
    elapsed = time.time() - t0

    cv2.putText(annotated, f"FPS: {1/elapsed:.1f}  Frame {i+1}/{NUM_FRAMES}",
                (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    _, enc = cv2.imencode('.jpg', annotated)
    clear_output(wait=True)
    display(IPyImage(data=enc.tobytes()))

    print(f"Frame {i+1}/{NUM_FRAMES}  |  {elapsed*1000:.0f} ms  |  {len(detections)} person(s)")
    for d in detections:
        print(f"  • {d['label'].replace('_', ' '):<32} {d['conf']:.1%}")

    if i < NUM_FRAMES - 1:
        time.sleep(FRAME_DELAY)

print("\n✅ Done.")

## 💾 Cell 14 — Export Models for Deployment

In [ ]:
import json

EXPORT_DIR = pathlib.Path('/content/export')
EXPORT_DIR.mkdir(exist_ok=True)

# ── 1. Save EfficientNet-B0 weights + class map ───────────────────────────────
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'class_to_idx'    : CLASS_TO_IDX,
        'idx_to_class'    : IDX_TO_CLASS,
        'num_classes'     : NUM_CLASSES,
        'img_size'        : IMG_SIZE,
    },
    EXPORT_DIR / 'efficientnet_b0_employee_activity.pth'
)

# ── 2. Export class map to JSON (human-readable) ──────────────────────────────
with open(EXPORT_DIR / 'class_map.json', 'w') as f:
    json.dump({'class_to_idx': CLASS_TO_IDX, 'idx_to_class': IDX_TO_CLASS}, f, indent=2)

# ── 3. Export EfficientNet to TorchScript (for C++/mobile deployment) ─────────
model.eval()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
traced = torch.jit.trace(model, dummy_input)
traced.save(str(EXPORT_DIR / 'efficientnet_b0_traced.pt'))

# ── 4. Export YOLOv8n to ONNX (optional — great for TensorRT) ─────────────────
# yolo_model.export(format='onnx', imgsz=640)

# ── 5. Zip and download everything ───────────────────────────────────────────
!cd /content && zip -r export_models.zip export/
files.download('/content/export_models.zip')

print("\n✅ Exported files:")
for f in EXPORT_DIR.iterdir():
    print(f"  {f.name:<45}  {f.stat().st_size/1e6:.1f} MB")

## 🚀 Cell 15 — Standalone Local Script (run outside Colab)

In [ ]:
# This cell writes a self-contained Python script you can run locally
# with: python run_live.py --source 0        (webcam)
#       python run_live.py --source video.mp4 (video file)

script = '''
#!/usr/bin/env python3
"""
Employee Activity Detection — Local Runtime
Usage:
    python run_live.py --source 0            # webcam
    python run_live.py --source video.mp4    # video file
    python run_live.py --source rtsp://...   # IP camera
"""
import argparse, json, time
import cv2, numpy as np
import torch, torch.nn as nn
import timm
from ultralytics import YOLO

# ── Config ────────────────────────────────────────────────────────────────────
CLASSIFIER_CKPT = 'efficientnet_b0_employee_activity.pth'
CLASS_MAP_JSON  = 'class_map.json'
IMG_SIZE        = 224
YOLO_CONF       = 0.40
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def load_classifier(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    num_classes = ckpt['num_classes']
    model = timm.create_model('efficientnet_b0', pretrained=False,
                               num_classes=0, global_pool='avg')
    model.classifier = nn.Sequential(
        nn.Dropout(0.3), nn.Linear(model.num_features, 512),
        nn.SiLU(),       nn.Dropout(0.2),
        nn.Linear(512, num_classes)
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval().to(device)
    return model, ckpt['idx_to_class']

def preprocess(crop_bgr, size):
    crop = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    crop = cv2.resize(crop, (size, size))
    t    = (crop / 255.0 - MEAN) / STD
    return torch.tensor(t, dtype=torch.float32).permute(2,0,1).unsqueeze(0)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--source', default='0')
    args = ap.parse_args()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device: {device}')

    yolo  = YOLO('yolov8n.pt')
    model, idx_to_class = load_classifier(CLASSIFIER_CKPT, device)

    np.random.seed(42)
    colors = {c: tuple(np.random.randint(50,255,3).tolist()) for c in idx_to_class.values()}

    src = int(args.source) if args.source.isdigit() else args.source
    cap = cv2.VideoCapture(src)

    prev_time = time.time()
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        H, W = frame.shape[:2]
        results = yolo(frame, classes=[0], conf=YOLO_CONF, verbose=False)

        for r in results:
            for box in r.boxes.xyxy.cpu().numpy().astype(int):
                x1,y1,x2,y2 = box
                crop = frame[max(0,y1-20):min(H,y2+20),
                             max(0,x1-20):min(W,x2+20)]
                if crop.size == 0: continue
                inp   = preprocess(crop, IMG_SIZE).to(device)
                with torch.no_grad():
                    prob  = torch.softmax(model(inp), 1)[0]
                    idx   = prob.argmax().item()
                label = idx_to_class[str(idx)]
                conf  = prob[idx].item()
                color = colors[label]
                cv2.rectangle(frame, (x1,y1),(x2,y2), color, 2)
                text = f"{label.replace('_',' ')} {conf:.0%}"
                (tw,th),_ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
                cv2.rectangle(frame,(x1,y1-th-8),(x1+tw+6,y1),color,-1)
                cv2.putText(frame,text,(x1+3,y1-4),
                            cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),2)

        fps = 1/(time.time()-prev_time); prev_time=time.time()
        cv2.putText(frame,f'FPS:{fps:.1f}',(10,30),
                    cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,255),2)
        cv2.imshow('Employee Activity Detection — press Q to quit', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == '__main__':
    main()
'''

with open('/content/run_live.py', 'w') as f:
    f.write(script)

files.download('/content/run_live.py')
print("✅ run_live.py downloaded. Run locally with:")
print("   python run_live.py --source 0         # webcam")
print("   python run_live.py --source video.mp4 # video file")
print("   python run_live.py --source rtsp://.. # IP camera")

## 📝 Cell 16 — Activity Log & Summary Stats

In [ ]:
# Run this after processing a video to get a time-aggregated activity report
# Requires `detections_log` list populated during video processing (Cell 12).
# Here we simulate with the validation set detections.

model.eval()
activity_counts = {cls: 0 for cls in CLASSES}

sample_for_report = val_split.sample(min(200, len(val_split)), random_state=42)
for fname in sample_for_report['filename']:
    frame = cv2.imread(str(TRAIN_IMG / fname))
    _, detections = annotate_frame(frame, yolo_model, model)
    for d in detections:
        if d['conf'] >= 0.4:
            activity_counts[d['label']] += 1

# ── Bar chart ─────────────────────────────────────────────────────────────────
sorted_counts = dict(sorted(activity_counts.items(), key=lambda x: x[1], reverse=True))
total = sum(sorted_counts.values()) or 1

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(
    [k.replace('_', '\n') for k in sorted_counts.keys()],
    sorted_counts.values(),
    color='steelblue'
)
for bar, count in zip(bars, sorted_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{count/total:.0%}", ha='center', va='bottom', fontsize=8)

ax.set_title('Activity Distribution — Detected in Sample Frames')
ax.set_ylabel('Detections')
plt.tight_layout()
plt.show()

print("\nActivity breakdown:")
for act, cnt in sorted_counts.items():
    bar = '█' * int(cnt / max(sorted_counts.values()) * 30)
    print(f"  {act:<35} {cnt:>4}  {cnt/total:>5.1%}  {bar}")